In [3]:
import os
import cv2
import xml.etree.ElementTree as ET
from mtcnn import MTCNN
import numpy as np

# 计算IoU
def calculate_iou(pred_box, gt_box):
    pred_xmin, pred_ymin, pred_xmax, pred_ymax = pred_box
    gt_xmin, gt_ymin, gt_xmax, gt_ymax = gt_box
    intersection_xmin = max(pred_xmin, gt_xmin)
    intersection_ymin = max(pred_ymin, gt_ymin)
    intersection_xmax = min(pred_xmax, gt_xmax)
    intersection_ymax = min(pred_ymax, gt_ymax)
    
    if intersection_xmax > intersection_xmin and intersection_ymax > intersection_ymin:
        intersection_area = (intersection_xmax - intersection_xmin) * (intersection_ymax - intersection_ymin)
    else:
        return 0  # 没有交集
    
    pred_area = (pred_xmax - pred_xmin) * (pred_ymax - pred_ymin)
    gt_area = (gt_xmax - gt_xmin) * (gt_ymax - gt_ymin)
    
    union_area = pred_area + gt_area - intersection_area
    
    return intersection_area / union_area

# 解析XML文件获取真实框
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    boxes = []
    for obj in root.findall("object"):
        xmin = int(obj.find("bndbox/xmin").text)
        ymin = int(obj.find("bndbox/ymin").text)
        xmax = int(obj.find("bndbox/xmax").text)
        ymax = int(obj.find("bndbox/ymax").text)
        boxes.append((xmin, ymin, xmax, ymax))
    
    return boxes

# 使用MTCNN进行人脸检测
def detect_faces(image_path):
    image = cv2.imread(image_path)
    detector = MTCNN()
    faces = detector.detect_faces(image)
    
    boxes = []
    for face in faces:
        xmin, ymin, width, height = face['box']
        xmax = xmin + width
        ymax = ymin + height
        boxes.append((xmin, ymin, xmax, ymax))
    
    return boxes

# 绘制并保存所有图片的真实框和预测框，同时计算并显示IoU
def draw_and_save_images(image_folder, annotation_folder, save_folder):
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    
    iou_list = []  # 用于存储每张图片的IoU值
    total_images = 0
    total_iou = 0
    
    # 遍历image_folder下的所有子文件夹
    for subfolder in os.listdir(image_folder):
        subfolder_path = os.path.join(image_folder, subfolder)
        
        if os.path.isdir(subfolder_path):
            # 遍历子文件夹中的所有图片文件
            for filename in os.listdir(subfolder_path):
                if filename.endswith('.jpg'):
                    image_path = os.path.join(subfolder_path, filename)
                    xml_path = os.path.join(annotation_folder, filename.replace('.jpg', '.xml'))
                    
                    if os.path.exists(xml_path):  # 确保XML文件存在
                        # 解析XML标签获取真实框
                        gt_boxes = parse_xml(xml_path)
                        
                        # 使用MTCNN进行人脸检测，获取预测框
                        pred_boxes = detect_faces(image_path)
                        
                        # 读取图像
                        image = cv2.imread(image_path)
                        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                        
                        # 计算IoU
                        iou_per_image = 0
                        valid_pairs = 0
                        for gt_box in gt_boxes:
                            for pred_box in pred_boxes:
                                iou = calculate_iou(pred_box, gt_box)
                                if iou > 0:  # 只计算有交集的框
                                    iou_per_image += iou
                                    valid_pairs += 1
                        if valid_pairs > 0:
                            average_iou_per_image = iou_per_image / valid_pairs
                            iou_list.append(average_iou_per_image)
                            total_iou += average_iou_per_image
                            total_images += 1
                            print(f"{filename}的IoU: {average_iou_per_image:.4f}")
                        
                        # 绘制真实框
                        for gt_box in gt_boxes:
                            xmin, ymin, xmax, ymax = gt_box
                            cv2.rectangle(image_rgb, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)  # 绿色的真实框
                        
                        # 绘制预测框
                        for pred_box in pred_boxes:
                            xmin, ymin, xmax, ymax = pred_box
                            cv2.rectangle(image_rgb, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)  # 红色的预测框
                        
                        # 保存图像到show文件夹
                        save_path = os.path.join(save_folder, filename)
                        image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)  # 转回BGR格式以保存
                        cv2.imwrite(save_path, image_bgr)
                        print(f"保存图像: {save_path}")
    
    if total_images > 0:
        average_iou = total_iou / total_images
        print(f"所有图像的平均IoU: {average_iou:.4f}")
    else:
        print("没有有效的IoU计算数据。")

# 示例用法
image_folder = './face_data1'  # 输入图片文件夹路径
annotation_folder = './output_annotations'  # 标签文件夹路径
save_folder = './show'  # 输出保存图片的文件夹路径

draw_and_save_images(image_folder, annotation_folder, save_folder)


0_Parade_marchingband_1_100.jpg的IoU: 0.7640
保存图像: ./show\0_Parade_marchingband_1_100.jpg
0_Parade_marchingband_1_1015.jpg的IoU: 0.7341
保存图像: ./show\0_Parade_marchingband_1_1015.jpg
0_Parade_marchingband_1_1018.jpg的IoU: 0.6746
保存图像: ./show\0_Parade_marchingband_1_1018.jpg
0_Parade_marchingband_1_1022.jpg的IoU: 0.7511
保存图像: ./show\0_Parade_marchingband_1_1022.jpg
0_Parade_marchingband_1_1030.jpg的IoU: 0.5406
保存图像: ./show\0_Parade_marchingband_1_1030.jpg
保存图像: ./show\0_Parade_marchingband_1_1031.jpg
0_Parade_marchingband_1_1038.jpg的IoU: 0.5052
保存图像: ./show\0_Parade_marchingband_1_1038.jpg
0_Parade_marchingband_1_1046.jpg的IoU: 0.6782
保存图像: ./show\0_Parade_marchingband_1_1046.jpg
0_Parade_marchingband_1_1047.jpg的IoU: 0.7544
保存图像: ./show\0_Parade_marchingband_1_1047.jpg
0_Parade_marchingband_1_1048.jpg的IoU: 0.8303
保存图像: ./show\0_Parade_marchingband_1_1048.jpg
0_Parade_marchingband_1_108.jpg的IoU: 0.0952
保存图像: ./show\0_Parade_marchingband_1_108.jpg
0_Parade_marchingband_1_112.jpg的IoU: 0.7899
保存图像